# Per-dataset sensitivity analyses (CCM revision)

Primary results in the paper are per-dataset; this notebook adds the per-dataset
sensitivity analyses requested in review, using **each dataset's own preprocessing**
(imported from its `*Util.py`).

| Analysis | Reviewer point | Datasets |
|---|---|---|
| IPW-weighted interaction test | R1 Major 1 | eICU, PMAP, MIMIC-IV (observational) |
| Lasso S-learner (penalized interactions) | R1 Major 2 | all 4 |
| GATES for neurologic outcome | R1 minor h | all 4 |
| Uniform |r|>0.7 TTM-collinearity filter | R1 minor d | eICU (confirm unchanged) |

Exposure definitions unchanged: eICU `treatment_hypothermia`; PMAP <36°C ≥12 consecutive h;
MIMIC-IV cooling device ≥12 consecutive h; HYPERION randomization.

Run on the cluster from this folder:
```bash
jupyter nbconvert --to notebook --execute perDatasetSensitivity.ipynb \
  --output perDatasetSensitivity.ipynb --ExecutePreprocessor.timeout=-1
```
Writes `per_dataset_results.csv` for the response-letter `[INSERT]` slots.


In [4]:
import os

In [5]:
# CONFIG
SEED = 42
TEST_SIZE = 0.30
N_GATES_GROUPS = 5
PS_CLIP = (0.05, 0.95)
CORR_THRESHOLD = 0.7
REPO_ROOT = os.path.abspath('..')

# Per-dataset wiring. dir = folder holding the *Util.py and predictor CSV (relative to REPO_ROOT).
# util_module / util_dir are import__ed after chdir. treatment/mortality/neuro = column names
# passed to getTrainTestFunctions(aPredictedColumn=..., aTreatmentColumn=...).
DATASETS = [
    dict(name='eICU',     folder='eICU',     util='eICUUtil',
         treatment='Hypothermia', mortality='DeathAtDischarge', neuro='LastMGCSPositive',
         observational=True),
    dict(name='PMAP',     folder='pmap',     util='PMAPUtil',
         treatment='hypothermia', mortality='death_at_disch', neuro='LastMGCSPositive',
         observational=True),
    dict(name='MIMIC-IV', folder='mimiciv',  util='MIMICUtil',
         treatment='hypothermia', mortality='death_at_disch', neuro='LastMGCSPositive',
         observational=True),
    dict(name='HYPERION', folder='hyperion', util='hyperion_utils',
         treatment='Hypothermia', mortality='hospital_mortality', neuro='CPC12',
         observational=False),  # randomized: no propensity/IPW, unweighted GATES (as Fig 3)
]
# analysisFunctions dir (sibling of the repo) that hyperion_utils imports at module load.
ANALYSIS_FUNCTIONS_DIR = os.path.abspath(os.path.join(REPO_ROOT, '..', 'analysisFunctions'))


In [6]:
import sys, importlib, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.linear_model import LogisticRegressionCV
from xgboost import XGBClassifier
from econml.dml import CausalForestDML

np.random.seed(SEED)
if ANALYSIS_FUNCTIONS_DIR not in sys.path:
    sys.path.insert(0, ANALYSIS_FUNCTIONS_DIR)   # for hyperion_utils' `from analysis_functions import *`
RESULTS = []


/home/mbranda1/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loader — import each dataset's own `getTrainTestFunctions`

`aTreatmentSplit=True` returns `(df, X_train, X_test, T_train, T_test, y_train, y_test)`
with the treatment column held out of X — exactly the inputs CausalForestDML needs. We
chdir into the dataset folder so the util's relative `pd.read_csv(...)` calls resolve.


In [7]:
def load_split(cfg, outcome_col):
    ds_dir = os.path.join(REPO_ROOT, cfg['folder'])
    if ds_dir not in sys.path:
        sys.path.insert(0, ds_dir)
    cwd = os.getcwd()
    try:
        os.chdir(ds_dir)
        mod = importlib.import_module(cfg['util'])
        importlib.reload(mod)  # ensure the right module if names shadow
        out = mod.getTrainTestFunctions(
            aPredictedColumn=outcome_col,
            aTreatmentColumn=cfg['treatment'],
            aTreatmentSplit=True, aSkipTemp=True, aTestSize=TEST_SIZE)
    finally:
        os.chdir(cwd)
    df, X_tr, X_te, T_tr, T_te, y_tr, y_te = out
    return (X_tr.reset_index(drop=True), X_te.reset_index(drop=True),
            pd.Series(np.asarray(T_tr)).astype(int),
            pd.Series(np.asarray(T_te)).astype(int),
            pd.Series(np.asarray(y_tr)).astype(int),
            pd.Series(np.asarray(y_te)).astype(int))


def preprocess(X_tr, X_te):
    """Scale numeric (>2 levels) on train, KNN-impute (k=10) on train; apply to test."""
    cols = list(X_tr.columns)
    num = [c for c in cols if X_tr[c].dropna().nunique() > 2]
    scaler = StandardScaler().fit(X_tr[num])
    imputer = KNNImputer(n_neighbors=10)

    def tf(X, fit=False):
        X = X.copy()
        if num:
            X[num] = scaler.transform(X[num])
        arr = imputer.fit_transform(X) if fit else imputer.transform(X)
        return pd.DataFrame(arr, columns=cols, index=X.index)

    return tf(X_tr, fit=True), tf(X_te)


## Shared machinery (identical to the pooled notebook)

`ipw_interaction_test` and `run_gates` are the copy-paste-ready functions referenced in the
response letter; they match the paper's GATES/propensity handling.


In [8]:
def fit_cf(X_tr, T_tr, y_tr):
    cf = CausalForestDML(
        model_y=XGBClassifier(max_depth=3, n_estimators=50),
        model_t=XGBClassifier(max_depth=2, n_estimators=20),
        discrete_treatment=True, discrete_outcome=True,
        random_state=SEED, n_jobs=-1)
    cf.fit(y_tr, T_tr, X=X_tr, cache_values=True)
    return cf


def cf_propensity(cf, X):
    preds = []
    for mc in cf.models_t:
        for mdl in mc:
            p = mdl.predict_proba(np.asarray(X))
            preds.append(p[:, 1] if p.ndim == 2 else np.ravel(p))
    return np.clip(np.mean(np.vstack(preds), axis=0), 1e-6, 1 - 1e-6)


def lrt_cate_interaction(y, T, cate):
    df = pd.DataFrame({'const': 1.0, 'T': np.asarray(T, float), 'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    y = np.asarray(y, float)
    if np.ptp(df['cate']) < 1e-12:      # CATE constant -> no interaction estimable
        return {'lr_stat': np.nan, 'p': np.nan, 'note': 'CATE constant (TTM unused)'}
    m0 = sm.Logit(y, df[['const', 'T', 'cate']]).fit(disp=False)
    m1 = sm.Logit(y, df[['const', 'T', 'cate', 'tx']]).fit(disp=False)
    lr = 2 * (m1.llf - m0.llf)
    return {'lr_stat': lr, 'p': chi2.sf(lr, 1), 'note': ''}


def ipw_interaction_test(y, T, cate, ps, clip=PS_CLIP):
    T = np.asarray(T, float); y = np.asarray(y, float)
    ps = np.clip(np.asarray(ps, float), *clip)
    w = np.where(T == 1, 1.0 / ps, 1.0 / (1.0 - ps))
    df = pd.DataFrame({'const': 1.0, 'T': T, 'cate': np.asarray(cate, float)})
    df['tx'] = df['T'] * df['cate']
    if np.ptp(df['cate']) < 1e-12:
        return {'wald_p': np.nan, 'weighted_lr_p': np.nan}
    fam = sm.families.Binomial()
    m0 = sm.GLM(y, df[['const', 'T', 'cate']], family=fam, var_weights=w).fit()
    m1 = sm.GLM(y, df[['const', 'T', 'cate', 'tx']], family=fam, var_weights=w).fit()
    dev = m0.deviance - m1.deviance
    return {'wald_p': m1.pvalues['tx'], 'weighted_lr_p': chi2.sf(dev, 1)}


def run_gates(cate, y, T, ps, observational, title=''):
    d = pd.DataFrame({'y': np.asarray(y, float), 'T': np.asarray(T, float),
                      'cate': np.asarray(cate, float)})
    if observational:
        ps = np.clip(np.asarray(ps, float), *PS_CLIP)
        d['w'] = np.where(d['T'] == 1, 1 / ps, 1 / (1 - ps))
    else:
        d['w'] = 1.0                      # HYPERION: unweighted OLS, as in Fig 3
    d['g'] = pd.qcut(d['cate'], q=N_GATES_GROUPS, labels=False, duplicates='drop') + 1
    rows = []
    for g, sub in d.groupby('g'):
        wls = sm.WLS(sub['y'], sm.add_constant(sub['T']), weights=sub['w']).fit()
        ci = np.asarray(wls.conf_int())
        rows.append({'group': int(g), 'n': len(sub), 'gate': wls.params.iloc[1],
                     'ci_low': ci[1, 0], 'ci_high': ci[1, 1]})
    gdf = pd.DataFrame(rows)
    gd = pd.get_dummies(d['g'].astype(int), prefix='g', drop_first=True).astype(float)
    Xr = sm.add_constant(pd.concat([d[['T']], gd], axis=1))
    Xf = Xr.copy()
    for c in gd.columns:
        Xf[f'T_x_{c}'] = d['T'].values * gd[c].values
    mf = sm.WLS(d['y'], Xf, weights=d['w']).fit()
    mr = sm.WLS(d['y'], Xr, weights=d['w']).fit()
    f_stat, f_p, _ = mf.compare_f_test(mr)
    rho, rho_p = stats.spearmanr(gdf['group'], gdf['gate'])
    fig, ax = plt.subplots(figsize=(5.5, 4))
    ax.errorbar(gdf['group'], gdf['gate'],
                yerr=[gdf['gate'] - gdf['ci_low'], gdf['ci_high'] - gdf['gate']],
                fmt='o-', capsize=5)
    ax.axhline(0, ls='--', color='gray')
    ax.set_xlabel('CATE quintile'); ax.set_ylabel('GATE')
    ax.set_title(f'GATES (neuro) — {title}\nF p={f_p:.3f}, rho={rho:.2f} (p={rho_p:.3f})')
    plt.tight_layout()
    plt.savefig(f"gates_neuro_{title.lower().replace(' ', '_').replace('-', '')}.png", dpi=150)
    plt.show()
    return gdf, {'f_p': f_p, 'spearman_rho': rho, 'spearman_p': rho_p}


def lasso_slearner(X_tr, T_tr, y_tr):
    Xd = X_tr.copy()
    Xd['TTM'] = np.asarray(T_tr, float)
    inter = []
    for c in X_tr.columns:
        ic = f'{c}_x_TTM'
        Xd[ic] = Xd[c] * Xd['TTM']
        inter.append(ic)
    clf = LogisticRegressionCV(penalty='l1', solver='saga', Cs=10, cv=5,
                               scoring='neg_log_loss', max_iter=5000,
                               n_jobs=-1, random_state=SEED)
    clf.fit(Xd, np.asarray(y_tr))
    coefs = pd.Series(clf.coef_[0], index=Xd.columns)
    nz = coefs[inter][coefs[inter] != 0]
    return len(nz), len(inter), float(clf.C_[0]), nz


## Run — datasets × outcomes

For each dataset and outcome: fit the dataset's own CausalForest, then LRT, IPW test
(observational only), GATES (neuro only, since Fig 3 already covers mortality GATES), and
the lasso S-learner. eICU additionally gets a uniform |r|>0.7 filter variant.


In [10]:
def run_one(cfg, outcome_key):
    outcome_col = cfg[outcome_key]
    label = f"{cfg['name']} / {outcome_key}"
    try:
        X_tr, X_te, T_tr, T_te, y_tr, y_te = load_split(cfg, outcome_col)
    except Exception as e:
        print(f"[SKIP] {label}: load failed ({type(e).__name__}: {e})")
        RESULTS.append({'dataset': cfg['name'], 'outcome': outcome_key, 'status': f'load-fail: {e}'})
        return
    Xtr, Xte = preprocess(X_tr, X_te)
    print(f"=== {label}: train {len(Xtr)} x {Xtr.shape[1]}, test {len(Xte)} ===")

    cf = fit_cf(Xtr, T_tr, y_tr)
    cate = np.ravel(cf.effect(Xte))
    lrt = lrt_cate_interaction(y_te, T_te, cate)
    row = {'dataset': cfg['name'], 'outcome': outcome_key, 'status': 'ok',
           'n_train': len(Xtr), 'n_features': Xtr.shape[1],
           'lrt_p': lrt['p'], 'lrt_note': lrt.get('note', '')}

    if cfg['observational']:
        ps = cf_propensity(cf, Xte)
        ipw = ipw_interaction_test(y_te, T_te, cate, ps)
        row['ipw_wald_p'] = ipw['wald_p']; row['ipw_lr_p'] = ipw['weighted_lr_p']
        print(f"  LRT p={lrt['p']}, IPW interaction Wald p={ipw['wald_p']}")
    else:
        ps = None
        print(f"  LRT p={lrt['p']} (randomized: no IPW)")

    if outcome_key == 'neuro':
        gdf, g = run_gates(cate, y_te, T_te, ps, cfg['observational'], title=cfg['name'])
        row['gates_neuro_f_p'] = g['f_p']
        row['gates_neuro_spearman_rho'] = g['spearman_rho']
        row['gates_neuro_spearman_p'] = g['spearman_p']

    n_nz, n_int, C, nz = lasso_slearner(Xtr, T_tr, y_tr)
    row['lasso_nonzero_interactions'] = n_nz; row['lasso_n_interactions'] = n_int
    row['lasso_C'] = C
    print(f"  lasso: {n_nz}/{n_int} interaction coefs survive (C={C:.4g})")
    RESULTS.append(row)


for cfg in DATASETS:
    for outcome_key in ['mortality', 'neuro']:
        run_one(cfg, outcome_key)


/home/mbranda1/ttmhte/eICU/eICUUtil.py:6: DtypeWarning: Columns (2188,2190) have mixed types. Specify dtype option on import or set low_memory=False.
  myPredictorsDf = pd.read_csv('eICUPredictorsDiag.csv')


ValueError: Shape of passed values is (1289, 1670), indices imply (1289, 1751)

## eICU uniform collinearity filter (R1 minor d)

Reviewer 1 asked why balance is tighter in PMAP/MIMIC (which drop covariates with
|r|>0.7 vs TTM) than in eICU (which does not). Here we apply the same filter to eICU and
confirm the HTE conclusion is unchanged.


In [2]:
def run_eicu_uniform_filter(outcome_key='mortality'):
    cfg = DATASETS[0]  # eICU
    X_tr, X_te, T_tr, T_te, y_tr, y_te = load_split(cfg, cfg[outcome_key])
    corr = X_tr.apply(lambda c: np.corrcoef(c.fillna(c.median()), T_tr)[0, 1]).abs()
    drop = corr[corr > CORR_THRESHOLD].index.tolist()
    print(f"eICU |r|>{CORR_THRESHOLD} with TTM -> dropping {len(drop)}: {drop}")
    keep = [c for c in X_tr.columns if c not in drop]
    Xtr, Xte = preprocess(X_tr[keep], X_te[keep])
    cf = fit_cf(Xtr, T_tr, y_tr)
    cate = np.ravel(cf.effect(Xte))
    lrt = lrt_cate_interaction(y_te, T_te, cate)
    ps = cf_propensity(cf, Xte)
    ipw = ipw_interaction_test(y_te, T_te, cate, ps)
    print(f"  after filter: LRT p={lrt['p']}, IPW Wald p={ipw['wald_p']}")
    RESULTS.append({'dataset': 'eICU (|r|>0.7 filter)', 'outcome': outcome_key, 'status': 'ok',
                    'n_features': len(keep), 'lrt_p': lrt['p'], 'ipw_wald_p': ipw['wald_p'],
                    'lrt_note': f'dropped {len(drop)} collinear'})

run_eicu_uniform_filter('mortality')


NameError: name 'DATASETS' is not defined

## Summary for the response letter

In [3]:
summary = pd.DataFrame(RESULTS)
summary.to_csv('per_dataset_results.csv', index=False)
display(summary.round(4))

print('\n--- Paste-ready ---')
ok = summary[summary['status'] == 'ok']
for _, r in ok.iterrows():
    bits = [f"LRT p={r.get('lrt_p')}"]
    if pd.notna(r.get('ipw_wald_p')):        bits.append(f"IPW p={r.get('ipw_wald_p'):.3f}")
    if pd.notna(r.get('gates_neuro_f_p')):   bits.append(f"GATES-neuro F p={r.get('gates_neuro_f_p'):.3f}")
    if pd.notna(r.get('lasso_nonzero_interactions')):
        bits.append(f"lasso {int(r['lasso_nonzero_interactions'])}/{int(r['lasso_n_interactions'])} interactions")
    print(f"{r['dataset']:24s} {r['outcome']:9s}: " + '; '.join(str(b) for b in bits))


NameError: name 'pd' is not defined